# Phase kickback and Deutsch–Jozsa
**Quantum Computing · Week 5 (Quantum protocols and algorithms) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w5d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Phase kickback** turns a function's answer into a sign, and **Deutsch–Jozsa** uses those signs
to decide "constant or balanced?" with a single question to the function.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorSampler

# a CNOT whose target is |->: watch what happens to the CONTROL
qc = QuantumCircuit(2)
qc.h(0)                 # control q0 in |+>
qc.x(1)
qc.h(1)                 # target q1 in |->
before = Statevector(qc)
qc.cx(0, 1)
after = Statevector(qc)

X0 = SparsePauliOp("IX")                 # X on qubit 0 (Qiskit reads Pauli strings right to left)
X1 = SparsePauliOp("XI")                 # X on qubit 1
print("<X> of the control: before", round(before.expectation_value(X0).real, 3), " after", round(after.expectation_value(X0).real, 3))
print("<X> of the target : before", round(before.expectation_value(X1).real, 3), " after", round(after.expectation_value(X1).real, 3))

The target didn't change, but the control flipped from $|+\rangle$ to $|-\rangle$: the CNOT acted
like a Z on the control. The "answer" was kicked back into the control's phase.

## Oracles from a truth table
An oracle for $f$ flips the output qubit exactly when $f(x) = 1$. For a small $n$ we can build one
from any truth table, one multi-controlled X per input with $f(x) = 1$.

In [ ]:
def oracle_from_table(table, n=3):
    """A gate on n input qubits plus an output qubit (number n): flips the output when table[x] = 1."""
    qc = QuantumCircuit(n + 1, name="U_f")
    for x, fx in enumerate(table):
        if fx:
            zeros = [q for q in range(n) if not (x >> q) & 1]   # bit q of x lives on qubit q
            if zeros:
                qc.x(zeros)                                    # so that this x looks like 11...1
            qc.mcx(list(range(n)), n)                          # flip the output if all inputs are 1
            if zeros:
                qc.x(zeros)                                    # undo
    return qc

def dj_circuit(oracle, n=3):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(n)                                                    # output qubit in |->: ready for kickback
    qc.h(range(n))                                             # all 2^n inputs at once
    qc.barrier()
    qc.compose(oracle, inplace=True)                           # ONE query
    qc.barrier()
    qc.h(range(n))                                             # interference
    qc.measure(range(n), range(n))
    return qc

parity = [bin(x).count("1") % 2 for x in range(8)]
dj_circuit(oracle_from_table(parity)).draw("mpl", style="clifford", fold=-1)

## Constant or balanced? One query each

In [ ]:
rng = np.random.default_rng(7)
functions = {
    "constant 0": [0] * 8,
    "constant 1": [1] * 8,
    "parity x0 ^ x1 ^ x2": parity,
    "majority of 3": [int(bin(x).count("1") >= 2) for x in range(8)],
    "random balanced": list(rng.permutation([0] * 4 + [1] * 4)),
}
sampler = StatevectorSampler(seed=1)
for name, table in functions.items():
    counts = sampler.run([dj_circuit(oracle_from_table(table))], shots=1000).result()[0].data.c.get_counts()
    verdict = "constant" if counts.get("000", 0) == 1000 else "balanced"
    print(f"{name:20s} table {''.join(map(str, table))}  ->  {counts}  ->  {verdict}")

## Your turn: build oracles with plain gates, and break the promise
1. Build the oracle for the balanced function $f(x) = x_0 \oplus x_2$ with just two CNOTs, and run Deutsch–Jozsa.
2. Build $f(x) = x_0 \text{ AND } x_1$ with one Toffoli (`qc.ccx`). It is neither constant nor
   balanced, so it breaks the promise. What does the algorithm say now?

In [ ]:
def oracle_xor02():
    qc = QuantumCircuit(4)
    # TODO: flip the output qubit 3 once for x0 and once for x2
    return qc

def oracle_and01():
    qc = QuantumCircuit(4)
    # TODO: flip the output qubit 3 when x0 and x1 are both 1
    return qc

for name, oracle in [("x0 xor x2", oracle_xor02()), ("x0 and x1", oracle_and01())]:
    counts = sampler.run([dj_circuit(oracle)], shots=1000).result()[0].data.c.get_counts()
    print(name, counts)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def oracle_xor02():
    qc = QuantumCircuit(4)
    qc.cx(0, 3)
    qc.cx(2, 3)
    return qc

def oracle_and01():
    qc = QuantumCircuit(4)
    qc.ccx(0, 1, 3)
    return qc

for name, oracle in [("x0 xor x2", oracle_xor02()), ("x0 and x1", oracle_and01())]:
    counts = sampler.run([dj_circuit(oracle)], shots=1000).result()[0].data.c.get_counts()
    print(name, dict(sorted(counts.items())))
print("x0 xor x2 never gives 000: balanced. Notice it always gives 101, the bits that f looks at (tomorrow's lesson!).")
print("x0 and x1 gives 000 about a quarter of the time: (1/8 * sum of the signs)^2 = (4/8)^2. Without the promise, one shot proves nothing.")